# **ULANGAN TENGAH SEMESTER**
KASUS 07 - Prediksi Diabetes (Pima)

**Nama  :** Muhammad Nazriel Zildjian

**Kelas :** TIF RP 24 A CID

**NIM   :** 25442011331

**STEP 1 :** Melakukan Persiapan Dan Upload Data

In [1]:
!pip -q install streamlit scikit-learn pandas joblib
import os
os.makedirs("models", exist_ok=True)
from google.colab import files
files.upload()   # pilih kasus07_pima_diabetes.csv

Saving kasus07_pima_diabetes.csv to kasus07_pima_diabetes.csv


{'kasus07_pima_diabetes.csv': b'npreg,glu,bp,skin,bmi,ped,age,type\n5,86,68,28,30.2,0.364,24,No\n7,195,70,33,25.1,0.163,55,Yes\n5,77,82,41,35.8,0.156,35,No\n0,165,76,43,47.9,0.259,26,No\n0,107,60,25,26.4,0.133,23,No\n5,97,76,27,35.6,0.378,52,Yes\n3,83,58,31,34.3,0.336,25,No\n1,193,50,16,25.9,0.655,24,No\n3,142,80,15,32.4,0.2,63,No\n2,128,78,37,43.3,1.224,31,Yes\n0,137,40,35,43.1,2.288,33,Yes\n9,154,78,30,30.9,0.164,45,No\n1,189,60,23,30.1,0.398,59,Yes\n12,92,62,7,27.6,0.926,44,Yes\n1,86,66,52,41.3,0.917,29,No\n4,99,76,15,23.2,0.223,21,No\n1,109,60,8,25.4,0.947,21,No\n11,143,94,33,36.6,0.254,51,Yes\n1,149,68,29,29.3,0.349,42,Yes\n0,139,62,17,22.1,0.207,21,No\n2,99,70,16,20.4,0.235,27,No\n1,100,66,29,32.0,0.444,42,No\n4,83,86,19,29.3,0.317,34,No\n0,101,64,17,21.0,0.252,21,No\n1,87,68,34,37.6,0.401,24,No\n9,164,84,21,30.8,0.831,32,Yes\n1,99,58,10,25.4,0.551,21,No\n0,140,65,26,42.6,0.431,24,Yes\n5,108,72,43,36.1,0.263,33,No\n2,110,74,29,32.4,0.698,27,No\n1,79,60,42,43.5,0.678,23,No\n3,148,

**STEP 2 :** Membuat Fitur.py


In [2]:
%%writefile fitur.py
import numpy as np
import pandas as pd

TARGET = "type"
FITUR_MENTAH = ["npreg", "glu", "bp", "skin", "bmi", "ped", "age"]
KOLOM_TIDAK_BOLEH_NOL = ["glu", "bp", "skin", "bmi"]
FITUR_MODEL = ["npreg", "glu", "bp", "skin", "bmi", "age", "log_ped",
               "preg_rate", "glu_x_bmi", "ped_x_age", "skin_bmi_ratio"]

def tambah_fitur(X):
    X = pd.DataFrame(X)[FITUR_MENTAH].astype(float).copy()
    for c in KOLOM_TIDAK_BOLEH_NOL:
        X.loc[X[c] <= 0, c] = np.nan
    X["preg_rate"] = X["npreg"] / X["age"]
    X["glu_x_bmi"] = X["glu"] * X["bmi"] / 100.0
    X["ped_x_age"] = X["ped"] * X["age"]
    X["skin_bmi_ratio"] = X["skin"] / X["bmi"]
    X["log_ped"] = np.log1p(X["ped"])
    X = X.drop(columns=["ped"])
    return X[FITUR_MODEL].replace([np.inf, -np.inf], np.nan)

Writing fitur.py


**STEP 3 :** Melakukan Audit Data

In [3]:
import pandas as pd
import fitur
df = pd.read_csv("kasus07_pima_diabetes.csv")
print(df.shape); print(df.dtypes)
print("Duplikat:", df.duplicated().sum())
print(df.isna().sum())
print({c: int((df[c] <= 0).sum()) for c in fitur.KOLOM_TIDAK_BOLEH_NOL})
print(df.describe().round(2))
print(df["type"].value_counts())
df = df.drop_duplicates().reset_index(drop=True)

(200, 8)
npreg      int64
glu        int64
bp         int64
skin       int64
bmi      float64
ped      float64
age        int64
type      object
dtype: object
Duplikat: 0
npreg    0
glu      0
bp       0
skin     0
bmi      0
ped      0
age      0
type     0
dtype: int64
{'glu': 0, 'bp': 0, 'skin': 0, 'bmi': 0}
        npreg     glu      bp    skin     bmi     ped     age
count  200.00  200.00  200.00  200.00  200.00  200.00  200.00
mean     3.57  123.97   71.26   29.22   32.31    0.46   32.11
std      3.37   31.67   11.48   11.72    6.13    0.31   10.98
min      0.00   56.00   38.00    7.00   18.20    0.08   21.00
25%      1.00  100.00   64.00   20.75   27.58    0.25   23.00
50%      2.00  120.50   70.00   29.00   32.80    0.37   28.00
75%      6.00  144.00   78.00   36.00   36.50    0.62   39.25
max     14.00  199.00  110.00   99.00   47.90    2.29   63.00
type
No     132
Yes     68
Name: count, dtype: int64


**STEP 4 :** Target dan Split

In [4]:
from sklearn.model_selection import train_test_split
y = (df["type"].str.strip().str.lower() == "yes").astype(int)   # 1 = diabetes
X = df[fitur.FITUR_MENTAH]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print(X_tr.shape, X_te.shape, y_tr.mean().round(3), y_te.mean().round(3))

(150, 7) (50, 7) 0.34 0.34


**STEP 5 :** Pipeline dan Cross-Validation

In [5]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, fbeta_score
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

def buat_pipeline(est):
    prep = ColumnTransformer([("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())]), fitur.FITUR_MODEL)])
    return Pipeline([("fitur", FunctionTransformer(fitur.tambah_fitur, validate=False)),
                     ("prep", prep), ("clf", est)])

kandidat = {
    "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                           class_weight="balanced", random_state=42, n_jobs=-1),
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision", "recall": "recall",
           "precision": "precision", "f2": make_scorer(fbeta_score, beta=2)}

hasil_cv = {}
for nama, est in kandidat.items():
    r = cross_validate(buat_pipeline(est), X_tr, y_tr, cv=cv, scoring=scoring)
    hasil_cv[nama] = {m: [round(float(r[f"test_{m}"].mean()), 4), round(float(r[f"test_{m}"].std()), 4)] for m in scoring}
    print(nama, {m: v[0] for m, v in hasil_cv[nama].items()})

terbaik = max(hasil_cv, key=lambda n: hasil_cv[n]["roc_auc"][0])
print("Model terpilih:", terbaik)

LogisticRegression {'roc_auc': 0.86, 'pr_auc': 0.7943, 'recall': 0.7321, 'precision': 0.6459, 'f2': 0.7072}
RandomForest {'roc_auc': 0.8612, 'pr_auc': 0.7811, 'recall': 0.6394, 'precision': 0.6599, 'f2': 0.6374}
Model terpilih: RandomForest


**STEP 6 :** Threshold (ini data dari train saja)

In [6]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import recall_score

TARGET_RECALL = 0.85
pipe = buat_pipeline(kandidat[terbaik])
oof = cross_val_predict(pipe, X_tr, y_tr, cv=5, method="predict_proba")[:, 1]

ts = np.arange(0.05, 0.96, 0.01)
ok = [t for t in ts if recall_score(y_tr, (oof >= t).astype(int)) >= TARGET_RECALL]
kandidat_t = ok if ok else list(ts)
threshold = round(float(max(kandidat_t, key=lambda t: fbeta_score(y_tr, (oof >= t).astype(int), beta=2))), 2)
print("Threshold:", threshold)

Threshold: 0.22


**STEP 7 :** Fit Final dan Evaluasi Test (Jalankan Sekali Saja)

In [7]:
from sklearn.metrics import (accuracy_score, precision_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix)
pipe.fit(X_tr, y_tr)
proba = pipe.predict_proba(X_te)[:, 1]
pred = (proba >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()

metrik_test = {
    "accuracy": accuracy_score(y_te, pred), "precision": precision_score(y_te, pred, zero_division=0),
    "recall": recall_score(y_te, pred), "specificity": tn / (tn + fp),
    "f1": f1_score(y_te, pred), "f2": fbeta_score(y_te, pred, beta=2),
    "roc_auc": roc_auc_score(y_te, proba), "pr_auc": average_precision_score(y_te, proba)}
metrik_test = {k: round(float(v), 4) for k, v in metrik_test.items()}
metrik_test["confusion_matrix"] = {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}
metrik_test["n_test"] = int(len(y_te))
print(metrik_test)

{'accuracy': 0.56, 'precision': 0.4286, 'recall': 0.8824, 'specificity': 0.3939, 'f1': 0.5769, 'f2': 0.7282, 'roc_auc': 0.754, 'pr_auc': 0.5715, 'confusion_matrix': {'TN': 13, 'FP': 20, 'FN': 2, 'TP': 15}, 'n_test': 50}


**STEP 8 :** Simpan dengan Joblib

In [8]:
import joblib, json
metadata = {
    "kasus": "Kasus 7 - Pima Diabetes", "model": terbaik, "threshold": threshold,
    "fitur_mentah": fitur.FITUR_MENTAH, "fitur_model": fitur.FITUR_MODEL,
    "median_train": X_tr.median().round(2).to_dict(),
    "rentang_train": {c: [float(X_tr[c].min()), float(X_tr[c].max())] for c in fitur.FITUR_MENTAH},
    "hasil_cv": hasil_cv, "metrik_test": metrik_test, "n_train": int(len(y_tr)),
}
joblib.dump({"pipeline": pipe, "metadata": metadata}, "models/pima_pipeline.joblib")
json.dump(metadata, open("models/metrik.json", "w"), indent=2)
print(os.listdir("models"))

['metrik.json', 'pima_pipeline.joblib']


**STEP 9 :** Latih.py

In [9]:
%%writefile latih.py
"""latih.py - Pelatihan, seleksi model (CV di train), evaluasi test SATU kali, simpan joblib.
Butuh fitur.py (fungsi tambah_fitur) di folder yang sama."""
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score, fbeta_score,
                             precision_score, recall_score, roc_auc_score, accuracy_score,
                             make_scorer)
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

import fitur

BASE = Path(__file__).resolve().parent
NAMA_CSV = "kasus07_pima_diabetes.csv"
# cari CSV di dataset/ (struktur folder tugas) atau di folder yang sama (Colab)
DATA = next((p for p in [BASE / "dataset" / NAMA_CSV, BASE / NAMA_CSV] if p.exists()), BASE / "dataset" / NAMA_CSV)
OUT = BASE / "models"
SEED = 42
TARGET_RECALL = 0.85  # kesalahan paling mahal: pasien diabetes terlewat (false negative)


def buat_pipeline(estimator):
    prep = ColumnTransformer([("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), fitur.FITUR_MODEL)])
    return Pipeline([
        ("fitur", FunctionTransformer(fitur.tambah_fitur, validate=False)),
        ("prep", prep),
        ("clf", estimator),
    ])


def pilih_threshold(y, proba):
    """Threshold dari prediksi out-of-fold di TRAIN: maksimalkan F2 dengan recall >= target."""
    best_t, best_f2 = 0.5, -1
    for t in np.arange(0.05, 0.96, 0.01):
        pred = (proba >= t).astype(int)
        if recall_score(y, pred) >= TARGET_RECALL:
            f2 = fbeta_score(y, pred, beta=2)
            if f2 > best_f2:
                best_t, best_f2 = float(t), f2
    if best_f2 < 0:  # target recall tak tercapai -> ambil F2 tertinggi
        ts = np.arange(0.05, 0.96, 0.01)
        best_t = float(ts[np.argmax([fbeta_score(y, (proba >= t).astype(int), beta=2) for t in ts])])
    return round(best_t, 2)


def main():
    if not DATA.exists():
        raise SystemExit(f"Dataset tidak ditemukan: {DATA}\nUnduh kasus07_pima_diabetes.csv ke folder dataset/.")
    df = pd.read_csv(DATA)
    print("=== AUDIT ===")
    print("Ukuran:", df.shape)
    print("Tipe data:", df.dtypes.astype(str).to_dict())
    print("Duplikat:", int(df.duplicated().sum()))
    print("Missing:", df.isna().sum().to_dict())
    print("Nilai <= 0 (tidak wajar):", {c: int((df[c] <= 0).sum()) for c in fitur.KOLOM_TIDAK_BOLEH_NOL})
    print("Distribusi target:", df[fitur.TARGET].value_counts().to_dict())
    df = df.drop_duplicates().reset_index(drop=True)

    y = (df[fitur.TARGET].astype(str).str.strip().str.lower() == "yes").astype(int)  # 1 = diabetes
    X = df[fitur.FITUR_MENTAH]

    # SPLIT DULU sebelum imputasi/scaling/encoding
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

    kandidat = {
        "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
        "RandomForest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                               class_weight="balanced", random_state=SEED, n_jobs=-1),
    }
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
    scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision", "recall": "recall",
               "precision": "precision", "f2": make_scorer(fbeta_score, beta=2)}

    hasil_cv = {}
    for nama, est in kandidat.items():
        r = cross_validate(buat_pipeline(est), X_tr, y_tr, cv=cv, scoring=scoring, n_jobs=1)
        hasil_cv[nama] = {m: [round(float(r[f"test_{m}"].mean()), 4), round(float(r[f"test_{m}"].std()), 4)]
                          for m in scoring}
        print(f"[CV] {nama}: " + ", ".join(f"{m}={v[0]:.3f}±{v[1]:.3f}" for m, v in hasil_cv[nama].items()))

    terbaik = max(hasil_cv, key=lambda n: hasil_cv[n]["roc_auc"][0])  # seleksi hanya dari CV train
    print("Model terpilih:", terbaik)

    pipe = buat_pipeline(kandidat[terbaik])
    oof = cross_val_predict(pipe, X_tr, y_tr, cv=5, method="predict_proba")[:, 1]
    threshold = pilih_threshold(y_tr.values, oof)
    print("Threshold (dari OOF train):", threshold)

    pipe.fit(X_tr, y_tr)

    # EVALUASI TEST - SATU KALI
    proba = pipe.predict_proba(X_te)[:, 1]
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    metrik_test = {
        "accuracy": accuracy_score(y_te, pred), "precision": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred), "specificity": tn / (tn + fp),
        "f1": f1_score(y_te, pred), "f2": fbeta_score(y_te, pred, beta=2),
        "roc_auc": roc_auc_score(y_te, proba), "pr_auc": average_precision_score(y_te, proba),
        "confusion_matrix": {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)},
        "n_test": int(len(y_te)),
    }
    metrik_test = {k: (round(float(v), 4) if not isinstance(v, (dict, int)) else v) for k, v in metrik_test.items()}
    print("=== TEST (sekali) ===")
    print(json.dumps(metrik_test, indent=2))

    X_ref = fitur.tambah_fitur(X_tr)
    metadata = {
        "kasus": "Kasus 7 - Pima Diabetes", "model": terbaik, "threshold": threshold,
        "fitur_mentah": fitur.FITUR_MENTAH, "fitur_model": fitur.FITUR_MODEL,
        "kelas": {0: "Tidak diabetes (No)", 1: "Diabetes (Yes)"},
        "median_train": X_tr.replace(0, np.nan).median().round(2).to_dict(),
        "rentang_train": {c: [float(X_tr[c].min()), float(X_tr[c].max())] for c in fitur.FITUR_MENTAH},
        "hasil_cv": hasil_cv, "metrik_test": metrik_test, "n_train": int(len(y_tr)),
        "kolom_dibuang": {"(tidak ada ID)": "dataset tidak memiliki kolom ID atau kolom pasca-kejadian"},
    }
    OUT.mkdir(exist_ok=True)
    joblib.dump({"pipeline": pipe, "metadata": metadata}, OUT / "pima_pipeline.joblib")
    (OUT / "metrik.json").write_text(json.dumps(metadata, indent=2, default=str))
    print("Tersimpan di models/pima_pipeline.joblib")


if __name__ == "__main__":
    main()

Writing latih.py


**STEP 10 :** Jalankan latih.py

In [10]:
!python latih.py

=== AUDIT ===
Ukuran: (200, 8)
Tipe data: {'npreg': 'int64', 'glu': 'int64', 'bp': 'int64', 'skin': 'int64', 'bmi': 'float64', 'ped': 'float64', 'age': 'int64', 'type': 'object'}
Duplikat: 0
Missing: {'npreg': 0, 'glu': 0, 'bp': 0, 'skin': 0, 'bmi': 0, 'ped': 0, 'age': 0, 'type': 0}
Nilai <= 0 (tidak wajar): {'glu': 0, 'bp': 0, 'skin': 0, 'bmi': 0}
Distribusi target: {'No': 132, 'Yes': 68}
[CV] LogisticRegression: roc_auc=0.860±0.060, pr_auc=0.794±0.081, recall=0.732±0.174, precision=0.646±0.108, f2=0.707±0.144
[CV] RandomForest: roc_auc=0.861±0.065, pr_auc=0.781±0.103, recall=0.639±0.158, precision=0.660±0.111, f2=0.637±0.137
Model terpilih: RandomForest
Threshold (dari OOF train): 0.22
=== TEST (sekali) ===
{
  "accuracy": 0.56,
  "precision": 0.4286,
  "recall": 0.8824,
  "specificity": 0.3939,
  "f1": 0.5769,
  "f2": 0.7282,
  "roc_auc": 0.754,
  "pr_auc": 0.5715,
  "confusion_matrix": {
    "TN": 13,
    "FP": 20,
    "FN": 2,
    "TP": 15
  },
  "n_test": 50
}
Tersimpan di models

**STEP 11 :** app.py


In [11]:
%%writefile app.py
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st
import fitur

MODEL = Path(__file__).resolve().parent / "models" / "pima_pipeline.joblib"
st.set_page_config(page_title="Prediksi Diabetes Pima", page_icon="🩺")

@st.cache_resource
def muat():
    return joblib.load(MODEL)

if not MODEL.exists():
    st.error("Model belum ada. Jalankan sel training dulu.")
    st.stop()

paket = muat()
pipe, meta = paket["pipeline"], paket["metadata"]
med, rng = meta["median_train"], meta["rentang_train"]

st.title("🩺 Prediksi Diabetes - Wanita Pima")
st.caption(f"Model: {meta['model']} | threshold keputusan: {meta['threshold']}")
tab1, tab2 = st.tabs(["Prediksi", "Metrik evaluasi"])

with tab1:
    c1, c2 = st.columns(2)
    npreg = c1.number_input("Jumlah kehamilan (npreg)", 0, 20, 2)
    glu = c1.number_input("Glukosa plasma OGTT (glu)", 0.0, 300.0, float(med["glu"]))
    bp = c1.number_input("Tekanan darah diastolik (bp)", 0.0, 200.0, float(med["bp"]))
    skin = c2.number_input("Lipatan kulit triceps (skin)", 0.0, 100.0, float(med["skin"]))
    bmi = c2.number_input("BMI", 0.0, 80.0, float(med["bmi"]))
    ped = c2.number_input("Fungsi silsilah diabetes (ped)", 0.0, 3.0, float(med["ped"]), step=0.01)
    age = c1.number_input("Usia (age)", 15, 100, int(med["age"]))
    st.caption("Isi 0 pada glu/bp/skin/bmi jika tidak diketahui; akan diimputasi median data latih.")
    if st.button("Prediksi", type="primary"):
        raw = pd.DataFrame([dict(npreg=npreg, glu=glu, bp=bp, skin=skin, bmi=bmi, ped=ped, age=age)])
        p = float(pipe.predict_proba(raw)[0, 1])
        positif = p >= meta["threshold"]
        (st.error if positif else st.success)(f"Prediksi: **{'DIABETES (Yes)' if positif else 'TIDAK DIABETES (No)'}**")
        st.metric("Probabilitas diabetes", f"{p:.1%}")
        st.progress(min(max(p, 0.0), 1.0))
        st.dataframe(pd.DataFrame({"Kelas": ["No", "Yes"], "Probabilitas": [1 - p, p]}), hide_index=True)
        st.caption("Alat bantu belajar, bukan diagnosis medis.")

with tab2:
    m = meta["metrik_test"]
    st.subheader(f"Hasil pada data test (n={m['n_test']})")
    a, b, c, d = st.columns(4)
    a.metric("Recall", m["recall"]); b.metric("Precision", m["precision"])
    c.metric("F2", m["f2"]); d.metric("ROC-AUC", m["roc_auc"])
    st.write(f"Accuracy {m['accuracy']} | Specificity {m['specificity']} | F1 {m['f1']} | PR-AUC {m['pr_auc']}")
    cm = m["confusion_matrix"]
    st.table(pd.DataFrame([[cm["TN"], cm["FP"]], [cm["FN"], cm["TP"]]],
                          index=["Aktual No", "Aktual Yes"], columns=["Prediksi No", "Prediksi Yes"]))
    st.subheader("Perbandingan model (CV di data train, mean ± std)")
    st.dataframe(pd.DataFrame({n: {k: f"{v[0]:.3f} ± {v[1]:.3f}" for k, v in r.items()}
                               for n, r in meta["hasil_cv"].items()}).T)

Writing app.py


** STEP 12 :** Menjalankan Aplikasi

In [12]:
import subprocess, time, re
!pkill -f streamlit
time.sleep(2)
subprocess.Popen("streamlit run app.py --server.port 8501 --server.headless true > /content/streamlit.log 2>&1", shell=True)
time.sleep(8)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
subprocess.Popen("./cloudflared tunnel --url http://localhost:8501 > /content/cf.log 2>&1", shell=True)
time.sleep(12)
print(re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cf.log").read()))

['https://courses-bat-alternatives-zip.trycloudflare.com']
